# SigLLM on Amazon-Book — full flow (data → MF → Q-Former stages → eval → ablation)

Focus metric: **uAUC** (per-user AUC). Baseline: **SeLLa-Rec** (arXiv:2504.10107),
which reuses the **CoLLM** Amazon-Book split — so we convert that split into this
repo's `*_ood2.pkl` schema and run the existing pipeline on it.

**Order:** run Section A once (data), then B (MF / CF substrate), then C–E (Q-Former
stages), then F (eval) and G (ablations). Training cells shell out to the existing
`sigllm.pipelines.*` entrypoints with `configs/config_amazon.yaml`.

## 0. Setup

Assumes the repo is at `REPO` and the conda/venv env is already installed
(mirror the env steps from `notebooks/SigLLM.ipynb` if needed).

### Clone the repository

Clones the repo into `/content/SigLLM` so the hard-coded paths in the cells below
stay valid. The next cell puts `src/` on `sys.path` (and `PYTHONPATH`) so both this
kernel and the `run(...)` subprocesses can `import sigllm.*` — there is no
`pip install` step (the package is used in-place from `src/`).

In [ ]:
# 0.1 — Clone the repo (Colab). For a private repo, paste a GitHub PAT
#        (https://github.com/settings/tokens, "repo" scope) when prompted.
import os
from getpass import getpass

GITHUB_OWNER = 'htainvn'                 # repo owner on GitHub
REPO_NAME    = 'RecLLM'                  # repo name on GitHub
BRANCH       = 'feat/multi-token-cf'     # branch to use
CLONE_DIR    = '/content/SigLLM'         # local path (keeps the /content/SigLLM paths below valid)

if not os.path.isdir(CLONE_DIR):
    token = getpass('GitHub Personal Access Token (PAT, blank for public clone): ').strip()
    auth = f'{token}@' if token else ''
    repo_url = f'https://{auth}github.com/{GITHUB_OWNER}/{REPO_NAME}.git'
    !git clone {repo_url} {CLONE_DIR}
else:
    print('Repo already present at', CLONE_DIR)

%cd {CLONE_DIR}
!git checkout {BRANCH}
!git pull origin {BRANCH}
!ls -a

In [ ]:
# 0.2 — Pin transformers. The InstructBlip Q-Former mask routing this repo relies on
#        broke in the ~4.53+ transformers refactor (crashes Stage 1 with a cross-attention
#        mask shape mismatch). 4.44.2 is pre-refactor AND supports Qwen2 (>=4.37) for Stage 3.
#        NOTE: no --no-deps -> pip also downgrades tokenizers to the <0.20 that 4.44.2 needs.
#        transformers has no torch dependency, so this won't touch your CUDA/torch install.
!pip install -q "transformers==4.44.2"
import transformers, tokenizers, os as _os
print("transformers:", transformers.__version__, "| tokenizers:", tokenizers.__version__)
_refactor = _os.path.join(_os.path.dirname(transformers.__file__), "utils", "output_capturing.py")
print("has refactor file (want False):", _os.path.exists(_refactor))
print(">>> This downgraded transformers -> RESTART THE RUNTIME before running anything below.")

In [ ]:
import os, sys, subprocess
REPO = '/content/SigLLM'            # repo root (matches CLONE_DIR above)
SRC  = os.path.join(REPO, 'src')
CFG  = os.path.join(REPO, 'configs/config_amazon.yaml')
if SRC not in sys.path:
    sys.path.insert(0, SRC)        # so `from sigllm.* import ...` works in THIS kernel
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
# so `python -m sigllm.*` works in the run(...) subprocesses (cwd=REPO, package lives in src/)
os.environ['PYTHONPATH'] = SRC + os.pathsep + os.environ.get('PYTHONPATH', '')

# sanity check: the package must be importable before running any cell below
import sigllm  # noqa: F401
print('sigllm OK from', os.path.dirname(sigllm.__file__))

def run(cmd):
    '''Run a shell command from REPO, streaming its output INTO the notebook cell.

    subprocess inherits the kernel's real stdout fd, which Jupyter/Colab does NOT
    capture (only sys.stdout is). So we pipe the child output and re-emit it through
    print(), line by line, so it shows live in the cell. Raises on non-zero exit.

    PYTHONUNBUFFERED=1 is forced in the child env: when stdout is a pipe (not a TTY),
    Python block-buffers it (~8KB), so prints/log lines arrive late and in bursts,
    making a long training epoch look "stuck". Unbuffered => truly live output.
    '''
    print('>>', cmd)
    child_env = dict(os.environ, PYTHONUNBUFFERED='1')
    proc = subprocess.Popen(
        cmd, shell=True, cwd=REPO, env=child_env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,                       # line-buffered text mode
    )
    for line in proc.stdout:                         # stream as it arrives
        print(line, end='')
        sys.stdout.flush()
    code = proc.wait()
    if code != 0:
        raise subprocess.CalledProcessError(code, cmd)
    return code

In [ ]:
# 0.3 — download the base LLM to the path config_amazon.yaml expects
#        (model.llm_model = /content/SigLLM/ckpt/llm/qwen2-7b-instruct). Stages 2 & 3
#        load it locally from here. Qwen2-7B-Instruct is public (no HF token needed).
#        ~15GB download; needs a GPU that fits a 7B model for training (A100/L4, not T4).
import os
LLM_DIR = os.path.join(REPO, 'ckpt/llm/qwen2-7b-instruct')
if os.path.exists(os.path.join(LLM_DIR, 'config.json')):
    print('LLM already present at', LLM_DIR)
else:
    from huggingface_hub import snapshot_download
    snapshot_download('Qwen/Qwen2-7B-Instruct', local_dir=LLM_DIR)
    print('downloaded LLM to', LLM_DIR)
print('contents:', sorted(os.listdir(LLM_DIR))[:12])

## A. Amazon-Book data prep (reuse CoLLM / SeLLa-Rec split)

1. Point the paths below at your downloaded CoLLM Amazon-Book files.
2. **Inspect** them to see the real columns.
3. Fill the `ColumnMap` to match those columns.
4. **Convert** to `*_ood2.pkl`, then tag warm/cold and build the Q-Former pkls.

> CoLLM data: https://github.com/zyang1580/CoLLM (see the `collm-datasets` link).

In [ ]:
# A.0 — download the CoLLM Amazon-Book split. CoLLM ships it as a plain zip IN ITS
#        GITHUB REPO (no Google Drive / credentials needed): collm-datasets/amazon_book.zip.
#        Skip this cell if you've already placed the files in data/raw/amazon-book/.
import os, zipfile
COLLM_DIR = os.path.join(REPO, 'data/raw/amazon-book')
os.makedirs(COLLM_DIR, exist_ok=True)

ZIP_URL = 'https://github.com/zyang1580/CoLLM/raw/main/collm-datasets/amazon_book.zip'
ZIP_PATH = os.path.join(COLLM_DIR, 'amazon_book.zip')

if not os.path.exists(ZIP_PATH):
    run(f'wget -q --show-progress -O {ZIP_PATH} {ZIP_URL}')
else:
    print('zip already downloaded:', ZIP_PATH)

with zipfile.ZipFile(ZIP_PATH) as z:
    print('--- archive contents ---')
    for n in z.namelist():
        print('  ', n)
    z.extractall(COLLM_DIR)

# Show the extracted tree so you can point PATHS (cell A.1) at the real files.
print('\n--- extracted under', COLLM_DIR, '---')
for root, _, files in os.walk(COLLM_DIR):
    for f in sorted(files):
        if f == 'amazon_book.zip':
            continue
        print('  ', os.path.relpath(os.path.join(root, f), COLLM_DIR))

In [ ]:
# A.1 — paths to the CoLLM/SeLLa-Rec Amazon-Book files you downloaded.
# These CoLLM files are ALREADY in this repo's *_ood2.pkl schema (uid, iid, label,
# timestamp, his, his_title, title, flag, not_cold). Titles are inline -> no
# item_text file; only `genres` is absent (books have none here -> handled).
COLLM_DIR = os.path.join(REPO, 'data/raw/amazon-book/book')   # <-- where the files are
OUT_DIR   = os.path.join(REPO, 'data/processed/amazon-book')
os.makedirs(OUT_DIR, exist_ok=True)

PATHS = {
    'train':     os.path.join(COLLM_DIR, 'train_ood2.pkl'),
    'valid':     os.path.join(COLLM_DIR, 'valid_ood2.pkl'),
    'test':      os.path.join(COLLM_DIR, 'test_ood2.pkl'),
    'item_text': None,   # titles are inline on the interaction frames
}

In [ ]:
# A.2 — inspect the real schema BEFORE converting.
from sigllm.datasets import inspect_collm_files
frames = inspect_collm_files(PATHS)

In [ ]:
# A.3 — column mapping. The CoLLM Amazon-Book files already use canonical names
# and carry history inline, so the defaults match as-is (genres is absent -> '').
from sigllm.datasets import ColumnMap
cmap = ColumnMap(
    uid='uid', iid='iid', label='label', rating=None,
    timestamp='timestamp', his='his', his_title='his_title',
    title='title', genres=None,   # no genres column in this split
)
item_text_map = ColumnMap()  # unused (item_text_path=None)

In [ ]:
# A.4 — convert to this repo's *_ood2.pkl schema.
from sigllm.datasets import build_amazon_book
train_, valid_, test_, users_map, items_map = build_amazon_book(
    train_path=PATHS['train'], valid_path=PATHS['valid'], test_path=PATHS['test'],
    out_dir=OUT_DIR,
    item_text_path=PATHS.get('item_text'),
    cmap=cmap, item_text_map=item_text_map,
    rating_threshold=4.0,
    reuse_ids=True,   # CoLLM ids are already contiguous (0=pad) + history is prebuilt
)

In [ ]:
# A.5 — warm/cold tagging (unchanged from ML-1M pipeline).
from sigllm.datasets import process_warm_cold
_ = process_warm_cold(data_dir=OUT_DIR + '/', min_user_inter=3, min_item_inter=3)

In [ ]:
# A.6 — derive user_num / item_num and persist for later cells.
import pandas as pd
def _counts(out_dir):
    fr = [pd.read_pickle(os.path.join(out_dir, f'{s}_ood2.pkl')) for s in ('train','valid','test')]
    u = max(int(f['uid'].max()) for f in fr) + 1
    i = max(int(f['iid'].max()) for f in fr) + 1
    return u, i
USER_NUM, ITEM_NUM = _counts(OUT_DIR)
print('USER_NUM =', USER_NUM, ' ITEM_NUM =', ITEM_NUM)
OVR = f'model.rec_config.user_num={USER_NUM} model.rec_config.item_num={ITEM_NUM}'
print('override string:', OVR)

In [ ]:
# A.7 — build Q-Former alignment pkls (uses item_noun='book', rich_item_text=True
#        from config_amazon.yaml -> CHANGE 2d).
run(f'python -m sigllm.pipelines.multimodal.build_qformer_dataset --cfg-path {CFG}')

## B. MF baseline (CF substrate)

Trains the matrix-factorization backbone the Q-Former reads from, and reports
its own Valid/Test **uAUC** (a pure-CF reference point). The MF trainer derives
user/item counts from the data automatically.

In [ ]:
run(f'python -m sigllm.pipelines.rec.train_rec_baseline --cfg-path {CFG}')

## C. Q-Former Stage 1 — representation (ITC / ITM / ITG + item-item)

Pass the data-derived `user_num`/`item_num` as overrides so the frozen MF loads
with matching dimensions.

**Compute note:** the stage-3 improvements (Q1 per-sample instructions, Q2 rank
aux head, Q3 prompt mixing) do NOT touch stages 1-2 — stage 1 never reads the
`instruction` field (its losses consume the item text). If you already have
stage-1/2 checkpoints, **skip cells C and D** and go straight to E.


In [ ]:
run(f'python -m sigllm.pipelines.multimodal.train_qformer_stage1_representation --cfg-path {CFG} --options {OVR}')

## D. Q-Former Stage 2 — generative pretraining (LLM frozen)

In [ ]:
run(f'python -m sigllm.pipelines.multimodal.train_qformer_stage2_generative --cfg-path {CFG} --options {OVR}')

## E. Q-Former Stage 3 — CoLLM 2-step (LoRA, then Q-Former+proj CIE)

Step 1 trains LoRA on the text-only prompt; Step 2 trains the Q-Former + projection
on the multi-token CF prompt with LoRA frozen. The Run-B flags (Q1 `instruction_mode:
target_title`, Q2 `rank_aux_weight`, Q3 `title_free_ratio`, `qformer_lr_mult: 3.0`)
live in `config_amazon.yaml` — no extra options needed here.

**GATE: do not run Step 2 until Step 1's valid uAUC reaches ~0.65+.** Step 1 is
functionally TALLRec, which scores 0.6921 on this exact split (SeLLa-Rec paper);
a step-1 stuck near 0.55 means the LoRA never learned the task and step 2 will
inherit it. Watch the `Evaluation metrics | ... uAUC=` lines per epoch.

During Step 2, also watch the `Q-Former standalone (rank aux head)` log line:
`qformer_uAUC` should climb above the MF baseline (0.5366) early — that is the
direct evidence the bridge extracts ranking signal.


In [ ]:
run(f'python -m sigllm.pipelines.multimodal.train_qformer_stage3_step1_lora --cfg-path {CFG} --options {OVR}')

In [ ]:
run(f'python -m sigllm.pipelines.multimodal.train_qformer_stage3_step2_cie --cfg-path {CFG} --options {OVR}')

## F. Evaluation — uAUC overall + warm / cold

The Stage-3 runner evaluates `test`, `test_warm`, `test_cold` and logs **AUC/uAUC**
for each. Re-run Step 2 with `evaluate=True` and a trained `ckpt` to evaluate only.
Read the per-split `uAUC=...` lines from the log; cold uAUC is the headline for the
cold/sparse regime.

In [ ]:
# Evaluate-only on the trained Step-2 model.
# The step-1 LoRA ckpt and this run's step-2 best ckpt are loaded TOGETHER
# automatically (each checkpoint is partial: _save_checkpoint strips frozen
# params, so step-1 holds only LoRA and step-2 only Q-Former/projection).
# To evaluate a different step-2 checkpoint, add model.ckpt=<path> -- it is
# combined with the step-1 LoRA ckpt, no longer silently overwritten.
run(f'python -m sigllm.pipelines.multimodal.train_qformer_stage3_step2_cie --cfg-path {CFG} --options {OVR} run.evaluate=True')


## G. Ablations (all read out as uAUC)

These isolate the Q-Former's contribution — the core concern from the report.

| Arm | What it tests | How |
|---|---|---|
| **Title-free prompt** (2a) | Does the Q-Former carry item identity? | notitle `prompt_path` + `instruction_mode=static` + `title_free_ratio=0` |
| **Ablate soft tokens** | Marginal value of CF tokens | `model.ablate_soft_tokens=True` |
| **cf_injection_mode=both** (2j) | Soft tokens + CoRA weight delta | `model.cf_injection_mode=both` |

**Important since Run B:** with `instruction_mode: target_title` the Q-Former sees
the target title through its instruction even when the LLM prompt is title-free.
The G.1 control below therefore ALSO flips `instruction_mode=static` and
`title_free_ratio=0` so the model is strictly title-free end to end.

The Q-Former's standalone `qformer_uAUC` (rank aux head) is logged at every eval
of the main run — no separate arm needed for it.

If the title-free uAUC collapses toward 0.5 while the title prompt scores high, the
LLM was leaning on plain-text titles, not the Q-Former.


In [ ]:
NOTITLE = '/content/SigLLM/prompts/qformer_prompt_book_mt_notitle.txt'

# G.1 STRICT title-free control (CHANGE 2a): no titles anywhere — LLM prompt is
# title-free AND the Q-Former instruction is static (else target_title mode
# would leak the title through the Q-Former's text branch). Mixing is moot
# when the main prompt is already title-free, so it is disabled.
run(f'python -m sigllm.pipelines.multimodal.train_qformer_stage3_step2_cie --cfg-path {CFG} --options {OVR} run.qformer_stage3_step2.prompt_path={NOTITLE} run.qformer_stage3_step2.output_dir=/content/SigLLM/ckpt/abl_notitle_amazon/ model.qformer_config.instruction_mode=static model.title_free_ratio=0')


In [ ]:
# G.2 ablate soft tokens at eval (loads step-1 LoRA + step-2 best ckpt
#     automatically, zeroes the soft tokens before injection)
run(f'python -m sigllm.pipelines.multimodal.train_qformer_stage3_step2_cie --cfg-path {CFG} --options {OVR} run.evaluate=True model.ablate_soft_tokens=True')


In [ ]:
# G.3 cf_injection_mode=both (CHANGE 2j)
run(f'python -m sigllm.pipelines.multimodal.train_qformer_stage3_step2_cie --cfg-path {CFG} --options {OVR} model.cf_injection_mode=both run.qformer_stage3_step2.output_dir=/content/SigLLM/ckpt/abl_both_amazon/')

## H. Results (fill in)

Reference points on this exact split (SeLLa-Rec paper, >20-interaction uAUC
protocol, Qwen2-7B): MF **0.5366** · TALLRec **0.6921** · CoLLM-MF **0.6957** ·
BinLLM 0.7073 · SeLLa-Rec **0.7459**. (CoLLM's own paper, no filter: CoLLM-MF 0.6225.)

| Model | Test uAUC | Warm uAUC | Cold uAUC |
|---|---|---|---|
| MF (CF only) — expect ~0.5366 | | | |
| Step 1 / LoRA text-only (≈ TALLRec, target ~0.69) | | | |
| Q-Former standalone (`qformer_uAUC`, rank aux head) | | | |
| Q-Former full (title prompt) | | | |
| Q-Former (strict title-free, 2a) | | | |
| Q-Former − soft tokens (ablate, G.2) | | | |
| Q-Former (both injection, 2j) | | | |

Key comparisons: full − ablated = Q-Former's marginal contribution;
standalone vs MF = signal the bridge extracts beyond the dot product.

_SeLLa-Rec baseline + SASRec substrate land in the next pass._
